# AI as a Mathematical Collaborator — Missing weather measurements

**UDSM MATH4SDG · Saturday 21 November 2026 · 60-minute workshop**

**Your challenge:** Use generative AI to build **two different models** for estimating missing daily temperatures at **both** of two imaginary weather stations. Compare the models, challenge AI's reasoning, choose a predictor, and test your choice.

The observations are **entirely synthetic**. They are not TMA measurements and cannot support conclusions about the climate in Tanzania. You may use this fictional dataset in your AI prompts. **Never upload restricted industry or personal data to an external AI assistant without authorization.**

Work alone or in pairs. The goal is not to find the fanciest algorithm: it is to **test what AI suggests using mathematical evidence**.


## 1 · Inspect two incomplete time series (about 9 minutes)

Each row of `ai_weather_development.csv` describes **one day**. It contains 270 consecutive days, **1 January–27 September 2025**, with:

| Column | Meaning |
|---|---|
| `date` | Day of observation |
| `station_a_temp_c` | Temperature at station A (°C), sometimes missing |
| `station_b_temp_c` | Temperature at station B (°C), sometimes missing |
| `humidity_pct` | Humidity (%), available each day |
| `wind_speed_m_s` | Wind speed (m/s), available each day |

A missing CSV value appears as `NaN` in pandas. Sometimes A is missing, sometimes B, and sometimes **both**. We want to predict missing temperatures at **either station**.

**Information rule:** For an estimate on day `t`, we may use observed measurements from **that day** (including the other station if available) and all **earlier** days. We may **not** use later measurements or temperatures deliberately hidden for evaluation. This is reconstruction with information available by the end of the day, not unrestricted interpolation between earlier and later measurements.

Open this notebook inside `notebooks/` in the extracted MATH4SDG repository. Keep the CSV files in `data/exercises/`.

### Read the data


In [ ]:
import pandas as pd
import numpy as np

file_path = "../data/exercises/ai_weather_development.csv"
development = pd.read_csv(file_path)

display(development.head(8))
print("Days:", len(development))
print("Missing temperatures at A:", development["station_a_temp_c"].isna().sum())
print("Missing temperatures at B:", development["station_b_temp_c"].isna().sum())

### Use earlier days for training, later days for validation

First, convert `date` from text into a date and sort chronologically. Then use the **first 210 days** (1 January–29 July) for training, and the **next 60 days** (30 July–27 September) for validation. This is about 78%/22%; it is a practical choice, **not** a universal rule. A larger training period gives more examples, while a larger validation period provides more reliable comparisons.

In the validation period, we have **deliberately hidden some temperatures that were originally observed**, in addition to genuinely missing readings. The separate file `ai_weather_validation_answers.csv` stores only these deliberately hidden values. This lets us measure error where a true answer exists. **We never grade predictions against genuinely unknown measurements.**

A further 60 later days are reserved for a final test. Its hidden temperatures are obfuscated; you will only recover them after choosing a model.


In [ ]:
development["date"] = pd.to_datetime(development["date"])
development = development.sort_values("date").reset_index(drop=True)

training_days = development.iloc[:210].copy()
validation_days = development.iloc[210:].copy()

print("Training days:", len(training_days))
print("Validation days:", len(validation_days))
print("Training period:", training_days["date"].min().date(), "to", training_days["date"].max().date())
print("Validation period:", validation_days["date"].min().date(), "to", validation_days["date"].max().date())


## 2 · Complete the series and prepare prediction cases (about 6 minutes)

**The main mathematical idea is a simple recursion.** For each day, we leave observed temperatures unchanged. If only one station has a temperature, we use that observation to fill the other station's gap. If **both** are missing, we copy the two completed values from **yesterday**. Those values may themselves have been estimated.

We assume that **both stations have observations on the first day**; our synthetic data satisfy this. We work from the first day to the last, so this procedure never looks into the future.

The function `complete` below is the whole recursion. It uses NumPy arrays rather than Pandas. Its output is a provisional completion of the history, **not** the students' AI-model predictions. We keep the original observations, including their gaps, unchanged.

In [ ]:
def complete(a, b):
    a = a.copy()
    b = b.copy()

    for t in range(1, len(a)):
        if np.isnan(a[t]) and np.isnan(b[t]):
            a[t] = a[t - 1]
            b[t] = b[t - 1]
        elif np.isnan(a[t]):
            a[t] = b[t]
        elif np.isnan(b[t]):
            b[t] = a[t]

    return a, b


### Arrange the numbers for the models

We will use the **same prediction function for both stations**. Each row in the table below is a *station–day case*. `station_code = 0` means station A, and `1` means B. The column `target_temp_c` is the observed temperature that the model learns to estimate.

The input columns are:

- `other_today_c`: the **observed** temperature at the other station on that day, if available.
- `own_yesterday_c` and `other_yesterday_c`: yesterday's **completed** temperatures, observed or provisionally estimated by `complete`.
- Humidity, wind, day of year and station identity.

The next function, `make_cases`, **only arranges the arrays into a table** that scikit-learn can use. You do not need to understand this table-building code before developing your models. In particular, `target_temp_c` is *not* given to a model when it predicts a hidden temperature.

In [ ]:
FEATURES = [
    "station_code", "other_today_c", "own_yesterday_c", "other_yesterday_c",
    "humidity_pct", "wind_speed_m_s", "day_of_year",
]


def make_cases(days):
    days = days.sort_values("date").reset_index(drop=True)
    a = days["station_a_temp_c"].to_numpy(dtype=float)
    b = days["station_b_temp_c"].to_numpy(dtype=float)

    a_completed, b_completed = complete(a, b)
    yesterday_a = np.concatenate(([np.nan], a_completed[:-1]))
    yesterday_b = np.concatenate(([np.nan], b_completed[:-1]))

    dates = days["date"].to_numpy()
    humidity = days["humidity_pct"].to_numpy()
    wind = days["wind_speed_m_s"].to_numpy()
    day_of_year = days["date"].dt.dayofyear.to_numpy()

    rows = []
    for t in range(len(days)):
        rows.append((dates[t], 0, a[t], b[t], yesterday_a[t], yesterday_b[t],
                     humidity[t], wind[t], day_of_year[t]))
        rows.append((dates[t], 1, b[t], a[t], yesterday_b[t], yesterday_a[t],
                     humidity[t], wind[t], day_of_year[t]))

    columns = ["date", "station_code", "target_temp_c", "other_today_c",
               "own_yesterday_c", "other_yesterday_c", "humidity_pct",
               "wind_speed_m_s", "day_of_year"]
    return pd.DataFrame(rows, columns=columns)


examples = make_cases(development)
display(examples.head(6))

### Baseline and mean absolute error

Our **baseline** follows exactly the same simple rule as before:

1. If the other station has an **observed temperature today**, use it.
2. Otherwise, use **yesterday's completed temperature at this station**, whether observed or estimated.

If neither is available (only possible at the start of an otherwise incomplete history), use the mean of the observed training temperatures. We use this baseline as a reference, not as a proposed optimal model.

We compare models using the **mean absolute error (MAE)**:

$$\operatorname{MAE}=\frac{1}{n}\sum_{i=1}^n |y_i-\widehat y_i|.$$

The function `score` below **calculates MAE**: for each evaluation case it gets one prediction, finds its absolute error, and then returns the average. Lower MAE is better; the unit is °C.

**No looking ahead:** For a case on day `t`, `score` supplies only **training observations dated earlier than `t`** and just the **current case's seven input features**. It does not pass future cases or the hidden answer to the predictor. The AI-generated functions should use only their `train_rows` and `new_rows` arguments—not global notebook variables or files. You need not study every line of `score` to use it.

In [ ]:
def baseline_predictor(train_rows, new_rows):
    other_today = new_rows["other_today_c"].to_numpy()
    own_yesterday = new_rows["own_yesterday_c"].to_numpy()
    prediction = np.where(np.isnan(other_today), own_yesterday, other_today)
    return np.where(np.isnan(prediction), train_rows["target_temp_c"].mean(), prediction)


def score(predictor, train_rows, evaluation_cases):
    errors = []
    for i in range(len(evaluation_cases)):
        day = evaluation_cases["date"].iloc[i]
        past = train_rows[train_rows["date"] < day]
        current = evaluation_cases[FEATURES].iloc[[i]]

        prediction = np.asarray(predictor(past, current), dtype=float)
        if prediction.shape != (1,) or not np.isfinite(prediction).all():
            raise ValueError("Return one finite temperature prediction for this case.")

        truth = evaluation_cases["truth_temp_c"].iloc[i]
        errors.append(abs(truth - prediction[0]))

    return float(np.mean(errors))


last_training_day = training_days["date"].max()
train_examples = examples[
    np.logical_and(
        examples["date"] <= last_training_day,
        examples["target_temp_c"].notna()
    )
].copy()

answers = pd.read_csv(
    "../data/exercises/ai_weather_validation_answers.csv", parse_dates=["date"]
)
validation_cases = examples.merge(
    answers, on=["date", "station_code"], how="inner", validate="one_to_one"
)

baseline_mae = score(baseline_predictor, train_examples, validation_cases)
print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print("Observed station-day training examples:", len(train_examples))
print("Number of deliberately hidden validation temperatures:", len(validation_cases))

## 3 · Ask AI for your first model (about 9 minutes)

Open ChatGPT or another AI assistant. You can adapt this prompt, but use the specified function name:

> We have incomplete daily temperature observations at **two** weather stations. Our aim is to predict missing temperatures at either station. Each training example has a station code (`0` or `1`), observed temperature target `target_temp_c`, and these numeric features: `station_code`, `other_today_c`, `own_yesterday_c`, `other_yesterday_c`, `humidity_pct`, `wind_speed_m_s`, `day_of_year`. Some features contain `NaN`. Same-day measurements are allowed only when actually observed; yesterday's completed temperatures may be observed or provisional baseline estimates, based only on current and earlier observations. Suggest a **simple** prediction model in Python/scikit-learn, and write `first_predict_temperature(train_rows, new_rows)`, returning one prediction per row of `new_rows`. The training rows contain `target_temp_c` but the new rows contain **only** the seven feature columns. Fit preprocessing and the model using only the supplied `train_rows` (which contain observations strictly earlier than the day being predicted). The notebook calls the function for **one case at a time**. Do not use global notebook variables, read CSV files, or access later days or hidden answers inside your predictor. Do not install packages. Briefly explain the model and its treatment of missing inputs.

Paste the AI-generated function in the next cell, run it, and then calculate its validation MAE. If it fails, ask AI to debug the error, but inspect the code before using it.


In [ ]:
# Paste the definition provided by AI here, then run the cell.
# def first_predict_temperature(train_rows, new_rows):
#     ...
#     return predictions


In [ ]:
first_model_mae = score(first_predict_temperature, train_examples, validation_cases)
print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print(f"First AI-model validation MAE: {first_model_mae:.3f} °C")


**Record your first experiment (edit this Markdown cell):**

- What did you ask AI, and which mathematical model did it propose?
- Did you need to correct the code?
- How does it handle missing `other_today_c` or provisionally estimated previous-day temperatures?
- What was its validation MAE compared with the baseline?

## 4 · Ask AI for a *different* model (about 9 minutes)

Now ask AI for a **different model family**, not merely different settings for your first model. Possibilities include **linear or ridge regression**, **support vector regression (SVR)**, **decision trees**, **random forests**, and **k-nearest-neighbors regression**. You can also propose a simple mathematical rule of your own. These are alternatives, not a prescribed ranking.

> Propose a mathematical model **substantially different from the first**. Explain why this model might be better or worse for two incomplete temperature time series, including occasions when both current temperatures are missing. Write `second_predict_temperature(train_rows, new_rows)` with exactly the same input and output convention as the first function. Handle missing features. The function will be called for one case at a time; fit only with the supplied `train_rows` and use only the supplied `new_rows`, without accessing notebook globals, files, future observations or hidden answers. Use only existing Python packages.

If you select SVR or nearest neighbors, ask AI about **feature scaling**. If you select a more flexible model, ask about **overfitting**. Evaluate the second model on **the same validation cases**, using exactly the same MAE function.


In [ ]:
# Paste the second AI-generated function here, then run this cell.
# def second_predict_temperature(train_rows, new_rows):
#     ...
#     return predictions


In [ ]:
second_model_mae = score(second_predict_temperature, train_examples, validation_cases)
print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
print(f"First AI-model validation MAE: {first_model_mae:.3f} °C")
print(f"Second AI-model validation MAE: {second_model_mae:.3f} °C")


## 5 · Ask AI to compare and criticize (about 8 minutes)

Only **after both models have been constructed** should you ask AI to compare them. For example:

> Compare the two models we just tested. What mathematical assumptions differ? Which model is more vulnerable to overfitting or missing measurements? Can either accidentally use future information or concealed temperatures? Propose **one specific, testable improvement**, and explain why it might *fail*. Do not claim that a lower validation MAE proves the model is generally better.

Discuss the answer with your partner. You may implement one improvement **if time allows**, using the same validation cases. Otherwise keep the two original models.

**Record a claim you checked independently:** What did AI claim? How did you test it? Did the evidence support it?


In [ ]:
# Optional: paste an improved predictor here if you have time.
# def improved_predict_temperature(train_rows, new_rows):
#     ...
#     return predictions
# improved_model_mae = score(improved_predict_temperature, train_examples, validation_cases)


## 6 · Choose a predictor *before* the final test (about 2 minutes)

Look at the validation MAEs. Choose a predictor from `baseline_predictor`, `first_predict_temperature`, `second_predict_temperature`, or your optional improved predictor. **Make your own choice.** A more complicated method is not necessarily better.

In the next cell, replace `FUNCTION_NAME` by the actual name of your chosen function and remove the leading `#`. **Do not put parentheses after the function name:** you are selecting the function, not calling it.

Write down your decision and your reason **before revealing final-test temperatures**.


In [ ]:
# Remove the # and replace FUNCTION_NAME with your chosen predictor.
# chosen_predict_temperature = FUNCTION_NAME


## 7 · Reveal the independent final test (about 5 minutes)

The new file `ai_weather_final.csv` contains 60 **later** days, incomplete observations at both stations, and columns `masked_a_c` / `masked_b_c` holding *obfuscated* temperatures for some deliberately concealed observations. Other missing temperatures have no known value and cannot be scored.

**Stop here until you have chosen your predictor.** The instructor will reveal a simple inverse transformation. Complete the function `deobfuscation(x)` below with that revealed mathematical expression. Until then, the function raises an error instead of silently producing incorrect scores.

The code evaluates only the deliberately concealed temperatures. For **each test case**, `score` fits the chosen modelling method using only **actually observed training examples from earlier dates** (the available development observations). It supplies only one station/day's features, never other test cases or hidden answers.


In [ ]:
final_days = pd.read_csv(
    "../data/exercises/ai_weather_final.csv", parse_dates=["date"]
)


def deobfuscation(x):
    # Replace this line with the function revealed by the instructor.
    raise NotImplementedError("Deobfuscation function has not yet been revealed")


In [ ]:
# Construct prediction cases using only observed current/past measurements and provisional past estimates.
all_days = pd.concat(
    [development, final_days[development.columns]], ignore_index=True
)
all_examples = make_cases(all_days)

final_examples = all_examples[
    all_examples["date"] >= final_days["date"].min()
].copy()
all_training_examples = examples[examples["target_temp_c"].notna()].copy()

# Recover only deliberately concealed answers (the instructor provides the function).
a_answers = final_days[["date", "masked_a_c"]].rename(
    columns={"masked_a_c": "masked_temp_c"}
).assign(station_code=0)
b_answers = final_days[["date", "masked_b_c"]].rename(
    columns={"masked_b_c": "masked_temp_c"}
).assign(station_code=1)
final_answers = pd.concat([a_answers, b_answers], ignore_index=True)
final_answers = final_answers.dropna(subset=["masked_temp_c"])
final_answers["truth_temp_c"] = deobfuscation(final_answers["masked_temp_c"])

final_cases = final_examples.merge(
    final_answers[["date", "station_code", "truth_temp_c"]],
    on=["date", "station_code"], how="inner", validate="one_to_one"
)

baseline_final_mae = score(baseline_predictor, all_training_examples, final_cases)
chosen_final_mae = score(chosen_predict_temperature, all_training_examples, final_cases)
print(f"Final-test baseline MAE: {baseline_final_mae:.3f} °C")
print(f"Final-test chosen-model MAE: {chosen_final_mae:.3f} °C")
print("Final-test temperatures evaluated:", len(final_cases))


## 8 · Compare results across the five groups (final 12 minutes)

Share results with the facilitator. Together we will fill in this table. All groups use the **same held-out cases and MAE definition**.

| Group | First model | Second model | Chosen predictor | Final-test MAE | Better than baseline? |
|---|---|---|---|---:|---|
| 1 | — | — | — | — | — |
| 2 | — | — | — | — | — |
| 3 | — | — | — | — | — |
| 4 | — | — | — | — | — |
| 5 | — | — | — | — | — |

**Common baseline final-test MAE:** ____ °C

**Discuss:**

1. Which different model families did AI suggest? Did groups choose different approaches?
2. Was the lowest validation error also the lowest final-test error?
3. What happened when the *other station's same-day temperature* was also missing?
4. Did an AI explanation survive an independent check? What claim did you reject or revise?
5. What could go wrong with real missing observations—for example, a storm that disrupts both stations for several days?

**Scientific caution:** These are made-up data. Evaluation on deliberately concealed observations does not prove that a model will work when real missingness is systematic, and one small workshop does not determine which AI prompting strategy is best. **Our goal is to practice mathematical judgment while using AI.**
